# Deploy Real-Time News Credibility Project to Google Cloud

This notebook is for deploying my **Real-Time News Credibility Scoring System** to Google Cloud.

The goal is to show that the project is not only running locally, but can also be packaged with Docker and deployed on cloud infrastructure.

In this notebook, I will:

1. configure Google Cloud,
2. check the local project files,
3. build a Docker image,
4. push the image to Artifact Registry,
5. deploy the FastAPI service to Cloud Run,
6. deploy the Streamlit UI to Cloud Run,
7. test the deployed API and UI.

This follows the same basic workflow from the course GCP notebooks, but adapted to my own MLOps project.

## 1. Basic project configuration

First, I define the Google Cloud project and region.

Important: `PROJECT_ID` must be the actual Google Cloud **Project ID**, not only the project name shown in the console.

For my current setup, I am using:

```text
graphic-outlook-489716-n6
```

In [1]:
from pathlib import Path
import json
import subprocess
import time
import requests

PROJECT_ID = "graphic-outlook-489716-n6"   # change this if your GCP project ID is different
REGION = "europe-west1"

REPO_NAME = "news-credibility-repo"
IMAGE_NAME = "news-credibility-app"
IMAGE_TAG = "v1"

API_SERVICE_NAME = "news-credibility-api"
UI_SERVICE_NAME = "news-credibility-ui"

IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/{IMAGE_NAME}:{IMAGE_TAG}"

print("Project ID:", PROJECT_ID)
print("Region:", REGION)
print("Docker image URI:", IMAGE_URI)

Project ID: graphic-outlook-489716-n6
Region: europe-west1
Docker image URI: europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app:v1


## 2. Helper function for running commands

In the course notebooks, we used many terminal commands.  
On Windows and Jupyter, some commands behave differently than Linux/macOS.

To make this notebook more reliable, I use a small Python helper function to run commands with `subprocess`.

In [2]:
def run_cmd(cmd, check=True):
    """Run a terminal command and print the output."""
    print("\n>>", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")

    return result

## 3. Check that the notebook is running from the project root

This notebook should be run from the root folder of my project, for example:

```text
D:\_HSLU\MLOPS\real-time-news-credibility
```

The folder should contain:

```text
src/
app/
requirements.txt
models/
```

In [3]:
PROJECT_ROOT = Path.cwd()
print("Current folder:", PROJECT_ROOT)

required_paths = [
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "app",
    PROJECT_ROOT / "requirements.txt",
]

for path in required_paths:
    assert path.exists(), f"Missing required path: {path}"

print("Project structure looks correct.")

Current folder: D:\_HSLU\MLOPS\real-time-news-credibility
Project structure looks correct.


## 4. Authenticate and set active Google Cloud project

This checks which Google account is active and sets the active GCP project.

If authentication is missing, run:

```bash
gcloud auth login
```

If Application Default Credentials are needed, run:

```bash
gcloud auth application-default login
```

In [4]:
GCLOUD = r"D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd"

run_cmd([GCLOUD, "auth", "list"], check=False)
run_cmd([GCLOUD, "config", "set", "project", PROJECT_ID])
run_cmd([GCLOUD, "config", "get-value", "project"])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd auth list
Credentialed Accounts

ACTIVE: *
ACCOUNT: nishant_s1@me.iitr.ac.in


To set the active account, run:
    $ gcloud config set account `ACCOUNT`



>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd config set project graphic-outlook-489716-n6
[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd config get-value project
graphic-outlook-489716-n6



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'config', 'get-value', 'project'], returncode=0, stdout='graphic-outlook-489716-n6\n', stderr='')

## 5. Fix quota project warning if needed

Sometimes Google Cloud shows a warning that the active project does not match the quota project.

This command updates the quota project for Application Default Credentials.

In [5]:
run_cmd([GCLOUD, "auth", "application-default", "set-quota-project", PROJECT_ID], check=False)


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd auth application-default set-quota-project graphic-outlook-489716-n6

Credentials saved to file: [C:\Users\LENOVO\AppData\Roaming\gcloud\application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "graphic-outlook-489716-n6" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'auth', 'application-default', 'set-quota-project', 'graphic-outlook-489716-n6'], returncode=0, stdout='', stderr='\nCredentials saved to file: [C:\\Users\\LENOVO\\AppData\\Roaming\\gcloud\\application_default_credentials.json]\n\nThese credentials will be used by any library that requests Application Default Credentials (ADC).\n\nQuota project "graphic-outlook-489716-n6" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.\n')

## 6. Enable required Google Cloud APIs

For this deployment I need:

- **Cloud Run**: to run the API and UI containers
- **Artifact Registry**: to store Docker images
- **Cloud Build**: used by Google Cloud build/deployment tools

If billing is not linked, enabling APIs may fail. In that case, billing must be fixed from the Google Cloud Console or by using the correct account.

In [6]:
run_cmd([
    GCLOUD, "services", "enable",
    "run.googleapis.com",
    "artifactregistry.googleapis.com",
    "cloudbuild.googleapis.com",
    "--project", PROJECT_ID
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd services enable run.googleapis.com artifactregistry.googleapis.com cloudbuild.googleapis.com --project graphic-outlook-489716-n6
Operation "operations/acat.p2-727182253496-a9bab090-ff60-496e-8c64-4ddecdde8adc" finished successfully.



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'services', 'enable', 'run.googleapis.com', 'artifactregistry.googleapis.com', 'cloudbuild.googleapis.com', '--project', 'graphic-outlook-489716-n6'], returncode=0, stdout='', stderr='Operation "operations/acat.p2-727182253496-a9bab090-ff60-496e-8c64-4ddecdde8adc" finished successfully.\n')

## 7. Check that the trained model exists

Cloud Run will not automatically see my local Docker volumes.

So the trained model file must be inside the Docker image:

```text
models/baseline_model.joblib
```

If the model does not exist, this notebook runs the local training pipeline first.

In [7]:
MODEL_PATH = PROJECT_ROOT / "models" / "baseline_model.joblib"

if MODEL_PATH.exists():
    print("Model found:", MODEL_PATH)
else:
    print("Model not found. Running training pipeline first...")
    run_cmd(["python", "-m", "src.ingestion.load_static_data"])
    run_cmd(["python", "-m", "src.features.build_features"])
    run_cmd(["python", "-m", "src.training.train_baseline"])

assert MODEL_PATH.exists(), "Model file was not created. Check the training pipeline."
print("Model is ready for deployment.")

Model found: D:\_HSLU\MLOPS\real-time-news-credibility\models\baseline_model.joblib
Model is ready for deployment.


## 8. Create a cloud-specific Dockerfile

For local development I used Docker Compose with volumes.

For Cloud Run, I need a self-contained container image.  
That means the image should include:

- source code,
- app code,
- requirements,
- trained model artifact.

So this notebook creates a separate `Dockerfile.gcp`.

In [8]:
dockerfile_cloud = r"""
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --upgrade pip && \
    pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/
COPY app/ ./app/
COPY models/ ./models/

ENV PYTHONPATH=/app

CMD ["uvicorn", "src.inference.api:app", "--host", "0.0.0.0", "--port", "8080"]
"""

dockerfile_path = PROJECT_ROOT / "Dockerfile.gcp"
dockerfile_path.write_text(dockerfile_cloud.strip() + "\n", encoding="utf-8")

print("Created Dockerfile.gcp")
print(dockerfile_path.read_text())

Created Dockerfile.gcp
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --upgrade pip && \
    pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/
COPY app/ ./app/
COPY models/ ./models/

ENV PYTHONPATH=/app

CMD ["uvicorn", "src.inference.api:app", "--host", "0.0.0.0", "--port", "8080"]



## 9. Create Artifact Registry repository

Artifact Registry stores the Docker image before it is deployed to Cloud Run.

If the repository already exists, the command may show an error. That is okay.

In [8]:
run_cmd([
    GCLOUD, "artifacts", "repositories", "create", REPO_NAME,
    "--repository-format=docker",
    f"--location={REGION}",
    f"--project={PROJECT_ID}"
], check=False)


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd artifacts repositories create news-credibility-repo --repository-format=docker --location=europe-west1 --project=graphic-outlook-489716-n6
ERROR: (gcloud.artifacts.repositories.create) ALREADY_EXISTS: the repository already exists



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'artifacts', 'repositories', 'create', 'news-credibility-repo', '--repository-format=docker', '--location=europe-west1', '--project=graphic-outlook-489716-n6'], returncode=1, stdout='', stderr='ERROR: (gcloud.artifacts.repositories.create) ALREADY_EXISTS: the repository already exists\n')

## 10. Configure Docker authentication

Docker needs permission to push images to Google Artifact Registry.

This command updates Docker authentication for the selected region.

In [9]:
run_cmd([GCLOUD, "auth", "configure-docker", f"{REGION}-docker.pkg.dev", "--quiet"])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd auth configure-docker europe-west1-docker.pkg.dev --quiet

{
  "credHelpers": {
    "europe-west1-docker.pkg.dev": "gcloud"
  }
}
Adding credentials for: europe-west1-docker.pkg.dev
gcloud credential helpers already registered correctly.



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'auth', 'configure-docker', 'europe-west1-docker.pkg.dev', '--quiet'], returncode=0, stdout='', stderr='WARNING: Your config file at [C:\\Users\\LENOVO\\.docker\\config.json] contains these credential helper entries:\n\n{\n  "credHelpers": {\n    "europe-west1-docker.pkg.dev": "gcloud"\n  }\n}\nAdding credentials for: europe-west1-docker.pkg.dev\ngcloud credential helpers already registered correctly.\n')

## 11. Build Docker image locally

This builds the Docker image using `Dockerfile.gcp`.

The image contains the FastAPI app, Streamlit app, source code, and trained model.

In [10]:
run_cmd([
    "docker", "build",
    "-f", "Dockerfile.gcp",
    "-t", IMAGE_URI,
    "."
])


>> docker build -f Dockerfile.gcp -t europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app:v1 .
#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.gcp
#1 transferring dockerfile: 367B 0.0s done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 ...

#3 [auth] library/python:pull token for registry-1.docker.io
#3 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 0.9s

#4 [internal] load .dockerignore
#4 transferring context: 303B 0.0s done
#4 DONE 0.1s

#5 [internal] load build context
#5 transferring context: 5.15kB 0.0s done
#5 DONE 0.1s

#6 [1/7] FROM docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0
#6 resolve docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0 0.1s done
#6 DONE 0.1s

#7 [2/7] WORKD

CompletedProcess(args=['docker', 'build', '-f', 'Dockerfile.gcp', '-t', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app:v1', '.'], returncode=0, stdout='', stderr='#0 building with "desktop-linux" instance using docker driver\n\n#1 [internal] load build definition from Dockerfile.gcp\n#1 transferring dockerfile: 367B 0.0s done\n#1 DONE 0.0s\n\n#2 [internal] load metadata for docker.io/library/python:3.11-slim\n#2 ...\n\n#3 [auth] library/python:pull token for registry-1.docker.io\n#3 DONE 0.0s\n\n#2 [internal] load metadata for docker.io/library/python:3.11-slim\n#2 DONE 0.9s\n\n#4 [internal] load .dockerignore\n#4 transferring context: 303B 0.0s done\n#4 DONE 0.1s\n\n#5 [internal] load build context\n#5 transferring context: 5.15kB 0.0s done\n#5 DONE 0.1s\n\n#6 [1/7] FROM docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0\n#6 resolve docker.io/library/python:3.11-slim@sha256:a3ab0

## 12. Test the API container locally

Before pushing to Google Cloud, I test the Docker image locally.

This is useful because if it fails locally, it will probably also fail on Cloud Run.

In [11]:
run_cmd(["docker", "rm", "-f", "news-credibility-api-test"], check=False)

run_cmd([
    "docker", "run", "-d",
    "--name", "news-credibility-api-test",
    "-p", "8080:8080",
    IMAGE_URI
])

time.sleep(8)

print("Testing /health endpoint...")
health = requests.get("http://localhost:8080/health", timeout=20)
print(health.status_code, health.text)

print("Testing /predict endpoint...")
payload = {"text": "The government confirmed the new economic policy in an official statement."}
pred = requests.post("http://localhost:8080/predict", json=payload, timeout=30)
print(pred.status_code, pred.text)

run_cmd(["docker", "rm", "-f", "news-credibility-api-test"], check=False)


>> docker rm -f news-credibility-api-test
Error response from daemon: No such container: news-credibility-api-test


>> docker run -d --name news-credibility-api-test -p 8080:8080 europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app:v1
cd9809e201812dffa54f582f18c4ac8db3477b044517a740ba31379d3a17e206

Testing /health endpoint...
404 {"detail":"Not Found"}
Testing /predict endpoint...
200 {"prediction_label":"real","confidence":0.5477,"credibility_score":54,"risk_level":"Medium"}

>> docker rm -f news-credibility-api-test
news-credibility-api-test



CompletedProcess(args=['docker', 'rm', '-f', 'news-credibility-api-test'], returncode=0, stdout='news-credibility-api-test\n', stderr='')

## 13. Push Docker image to Artifact Registry

After local testing, I push the image to Google Cloud Artifact Registry.

In [10]:
run_cmd(["docker", "push", IMAGE_URI])


>> docker push europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app:v1
The push refers to repository [europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app]
797d495f2c68: Waiting
ec38dde27d2d: Waiting
3e31c6fa7f13: Waiting
acfe05beeb3a: Waiting
5b4d6ff92fc4: Waiting
8649771fee17: Waiting
fdb1961ee291: Waiting
69caaf1573d1: Waiting
0d0877382cbe: Waiting
4b35e2daa475: Waiting
45006ceeeea9: Waiting
45006ceeeea9: Waiting
797d495f2c68: Waiting
ec38dde27d2d: Waiting
3e31c6fa7f13: Waiting
acfe05beeb3a: Waiting
5b4d6ff92fc4: Waiting
8649771fee17: Waiting
fdb1961ee291: Waiting
69caaf1573d1: Waiting
0d0877382cbe: Waiting
4b35e2daa475: Waiting
4b35e2daa475: Waiting
45006ceeeea9: Waiting
797d495f2c68: Waiting
ec38dde27d2d: Waiting
3e31c6fa7f13: Waiting
acfe05beeb3a: Waiting
5b4d6ff92fc4: Waiting
8649771fee17: Waiting
fdb1961ee291: Waiting
69caaf1573d1: Waiting
0d0877382cbe: Waiting
4b35e2daa475: Waiting
45006ce

CompletedProcess(args=['docker', 'push', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app:v1'], returncode=0, stdout='The push refers to repository [europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app]\n797d495f2c68: Waiting\nec38dde27d2d: Waiting\n3e31c6fa7f13: Waiting\nacfe05beeb3a: Waiting\n5b4d6ff92fc4: Waiting\n8649771fee17: Waiting\nfdb1961ee291: Waiting\n69caaf1573d1: Waiting\n0d0877382cbe: Waiting\n4b35e2daa475: Waiting\n45006ceeeea9: Waiting\n45006ceeeea9: Waiting\n797d495f2c68: Waiting\nec38dde27d2d: Waiting\n3e31c6fa7f13: Waiting\nacfe05beeb3a: Waiting\n5b4d6ff92fc4: Waiting\n8649771fee17: Waiting\nfdb1961ee291: Waiting\n69caaf1573d1: Waiting\n0d0877382cbe: Waiting\n4b35e2daa475: Waiting\n4b35e2daa475: Waiting\n45006ceeeea9: Waiting\n797d495f2c68: Waiting\nec38dde27d2d: Waiting\n3e31c6fa7f13: Waiting\nacfe05beeb3a: Waiting\n5b4d6ff92fc4: Waiting\n8649771fee17: Waiting\nfdb1961ee291:

## 14. Verify image upload

This lists the Docker images stored in the Artifact Registry repository.

In [11]:
run_cmd([
    GCLOUD, "artifacts", "docker", "images", "list",
    f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}",
    f"--project={PROJECT_ID}"
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd artifacts docker images list europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo --project=graphic-outlook-489716-n6
IMAGE: europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app
DIGEST: sha256:4409d3e98cf7613a2c6d6f3ad493515303348a7fd0fcd76c43d374c5a15ad542
CREATE_TIME: 2026-05-31T16:45:04
UPDATE_TIME: 2026-05-31T16:45:04
SIZE: 1487

IMAGE: europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app
DIGEST: sha256:8fe6663aed6e82454f822c9031f6a7beb887cb9707238a83248dea4a7c9714cf
CREATE_TIME: 2026-05-30T17:41:03
UPDATE_TIME: 2026-05-31T16:45:04
SIZE: None

IMAGE: europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app
DIGEST: sha256:c1bbe69bb911b7cb0db4ed3b4a448bcbebe8133d2315ce2cdf53da94f1cbe055
CREATE_TIME: 2026-05-31T16:45:04
UPDATE_TIME: 2026-05-31T16:45:04
SIZE: 4726731760

IMAGE: europe-west

CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'artifacts', 'docker', 'images', 'list', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo', '--project=graphic-outlook-489716-n6'], returncode=0, stdout='IMAGE: europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app\nDIGEST: sha256:4409d3e98cf7613a2c6d6f3ad493515303348a7fd0fcd76c43d374c5a15ad542\nCREATE_TIME: 2026-05-31T16:45:04\nUPDATE_TIME: 2026-05-31T16:45:04\nSIZE: 1487\n\nIMAGE: europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app\nDIGEST: sha256:8fe6663aed6e82454f822c9031f6a7beb887cb9707238a83248dea4a7c9714cf\nCREATE_TIME: 2026-05-30T17:41:03\nUPDATE_TIME: 2026-05-31T16:45:04\nSIZE: None\n\nIMAGE: europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app\nDIGEST: sha256:c1bbe69bb911b7cb0db4ed3b4a448bcbebe8133d2315ce2cdf53da94f1cbe055\nCREATE_TIME: 2026-05

## 15. Deploy FastAPI service to Cloud Run

Now I deploy the FastAPI prediction API.

The API exposes endpoints such as:

```text
/health
/predict
```

For this university demo, I use `--allow-unauthenticated` so the endpoint is public.  
For production, authentication should be added.

In [12]:
run_cmd([
    GCLOUD, "run", "deploy", API_SERVICE_NAME,
    "--image", IMAGE_URI,
    "--region", REGION,
    "--project", PROJECT_ID,
    "--platform", "managed",
    "--allow-unauthenticated",
    "--port", "8080",
    "--memory", "2Gi",
    "--max-instances", "2",
    "--command", "uvicorn",
    "--args", "src.inference.api:app,--host,0.0.0.0,--port,8080"
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run deploy news-credibility-api --image europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app:v1 --region europe-west1 --project graphic-outlook-489716-n6 --platform managed --allow-unauthenticated --port 8080 --memory 2Gi --max-instances 2 --command uvicorn --args src.inference.api:app,--host,0.0.0.0,--port,8080
Deploying container to Cloud Run service [news-credibility-api] in project [graphic-outlook-489716-n6] region [europe-west1]
Deploying...
Setting IAM Policy................done
Creating Revision...............................................................................................................................................................................................................................................................................................................................................................................................................

CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'run', 'deploy', 'news-credibility-api', '--image', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-app:v1', '--region', 'europe-west1', '--project', 'graphic-outlook-489716-n6', '--platform', 'managed', '--allow-unauthenticated', '--port', '8080', '--memory', '2Gi', '--max-instances', '2', '--command', 'uvicorn', '--args', 'src.inference.api:app,--host,0.0.0.0,--port,8080'], returncode=0, stdout='', stderr='Deploying container to Cloud Run service [news-credibility-api] in project [graphic-outlook-489716-n6] region [europe-west1]\nDeploying...\nSetting IAM Policy................done\nCreating Revision.............................................................................................................................................................................................................................................................................

## 16. Get and test FastAPI Cloud Run URL

After deployment, I retrieve the public Cloud Run URL and test the API.

In [27]:
result = run_cmd([
    GCLOUD, "run", "services", "describe", API_SERVICE_NAME,
    "--region", REGION,
    "--project", PROJECT_ID,
    "--format", "json"
])

api_info = json.loads(result.stdout)
API_BASE_URL = api_info["status"]["url"]
PREDICT_URL = API_BASE_URL + "/predict/"

print("API base URL:", API_BASE_URL)
print("Predict URL:", PREDICT_URL)

print("Testing deployed /health...")
print(requests.get(API_BASE_URL + "/health", timeout=30).text)

print("Testing deployed /predict...")
print(requests.post(PREDICT_URL, json={
    "text": "The government confirmed the new policy in an official statement."
}, timeout=30).text)


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run services describe news-credibility-api --region europe-west1 --project graphic-outlook-489716-n6 --format json
{
  "apiVersion": "serving.knative.dev/v1",
  "kind": "Service",
  "metadata": {
    "annotations": {
      "run.googleapis.com/client-name": "gcloud",
      "run.googleapis.com/client-version": "569.0.0",
      "run.googleapis.com/ingress": "all",
      "run.googleapis.com/ingress-status": "all",
      "run.googleapis.com/maxScale": "20",
      "run.googleapis.com/operation-id": "e1a2ad61-1c50-47b2-8b3d-ccfa4bdfb43b",
      "run.googleapis.com/urls": "[\"https://news-credibility-api-727182253496.europe-west1.run.app\",\"https://news-credibility-api-xnipts7f7q-ew.a.run.app\"]",
      "serving.knative.dev/creator": "nishant_s1@me.iitr.ac.in",
      "serving.knative.dev/lastModifier": "nishant_s1@me.iitr.ac.in"
    },
    "creationTimestamp": "2026-05-30T15:41:47.520226Z",
    "generation": 2,
    "labels": {
      "cloud.goog

## 17. Deploy Streamlit UI to Cloud Run

Next, I deploy the Streamlit UI.

The UI needs to call the deployed FastAPI service, so I pass the API prediction endpoint as an environment variable:

```text
API_URL=<FastAPI Cloud Run URL>/predict
```

In [25]:
UI_IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/news-credibility-ui:v1"

run_cmd([
    "docker", "build",
    "-f", "Dockerfile.gcp.ui",
    "-t", UI_IMAGE_URI,
    "."
])


>> docker build -f Dockerfile.gcp.ui -t europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui:v1 .
#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.gcp.ui
#1 transferring dockerfile: 521B 0.0s done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 ...

#3 [auth] library/python:pull token for registry-1.docker.io
#3 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 1.0s

#4 [internal] load .dockerignore
#4 transferring context: 303B 0.0s done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 8.20kB 0.0s done
#5 DONE 0.1s

#6 [1/8] FROM docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0
#6 resolve docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0 0.1s done
#6 DONE 0.1s

#7 [2/8] 

CompletedProcess(args=['docker', 'build', '-f', 'Dockerfile.gcp.ui', '-t', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui:v1', '.'], returncode=0, stdout='', stderr='#0 building with "desktop-linux" instance using docker driver\n\n#1 [internal] load build definition from Dockerfile.gcp.ui\n#1 transferring dockerfile: 521B 0.0s done\n#1 DONE 0.0s\n\n#2 [internal] load metadata for docker.io/library/python:3.11-slim\n#2 ...\n\n#3 [auth] library/python:pull token for registry-1.docker.io\n#3 DONE 0.0s\n\n#2 [internal] load metadata for docker.io/library/python:3.11-slim\n#2 DONE 1.0s\n\n#4 [internal] load .dockerignore\n#4 transferring context: 303B 0.0s done\n#4 DONE 0.0s\n\n#5 [internal] load build context\n#5 transferring context: 8.20kB 0.0s done\n#5 DONE 0.1s\n\n#6 [1/8] FROM docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0\n#6 resolve docker.io/library/python:3.11-slim@sha256:

### Push UI image

In [26]:
run_cmd(["docker", "push", UI_IMAGE_URI])


>> docker push europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui:v1
The push refers to repository [europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui]
97115fa1548b: Waiting
8649771fee17: Waiting
3e31c6fa7f13: Waiting
c22b6a592a92: Waiting
69caaf1573d1: Waiting
797d495f2c68: Waiting
45006ceeeea9: Waiting
5b4d6ff92fc4: Waiting
13f2d8937887: Waiting
0ba6a6381680: Waiting
db9632e20c90: Waiting
5aea15a6b8a4: Waiting
45006ceeeea9: Waiting
5b4d6ff92fc4: Waiting
13f2d8937887: Waiting
0ba6a6381680: Waiting
db9632e20c90: Waiting
5aea15a6b8a4: Waiting
97115fa1548b: Waiting
8649771fee17: Waiting
3e31c6fa7f13: Waiting
c22b6a592a92: Waiting
69caaf1573d1: Waiting
797d495f2c68: Waiting
8649771fee17: Waiting
3e31c6fa7f13: Waiting
c22b6a592a92: Waiting
69caaf1573d1: Waiting
797d495f2c68: Waiting
45006ceeeea9: Waiting
5b4d6ff92fc4: Waiting
13f2d8937887: Waiting
0ba6a6381680: Waiting
db9632e20c90: Waiting
5aea15a6b

CompletedProcess(args=['docker', 'push', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui:v1'], returncode=0, stdout='The push refers to repository [europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui]\n97115fa1548b: Waiting\n8649771fee17: Waiting\n3e31c6fa7f13: Waiting\nc22b6a592a92: Waiting\n69caaf1573d1: Waiting\n797d495f2c68: Waiting\n45006ceeeea9: Waiting\n5b4d6ff92fc4: Waiting\n13f2d8937887: Waiting\n0ba6a6381680: Waiting\ndb9632e20c90: Waiting\n5aea15a6b8a4: Waiting\n45006ceeeea9: Waiting\n5b4d6ff92fc4: Waiting\n13f2d8937887: Waiting\n0ba6a6381680: Waiting\ndb9632e20c90: Waiting\n5aea15a6b8a4: Waiting\n97115fa1548b: Waiting\n8649771fee17: Waiting\n3e31c6fa7f13: Waiting\nc22b6a592a92: Waiting\n69caaf1573d1: Waiting\n797d495f2c68: Waiting\n8649771fee17: Waiting\n3e31c6fa7f13: Waiting\nc22b6a592a92: Waiting\n69caaf1573d1: Waiting\n797d495f2c68: Waiting\n45006ceeeea9: Waiting\n5b4d6ff92fc4: W

### Deploy UI

In [28]:
run_cmd([
    GCLOUD, "run", "deploy", UI_SERVICE_NAME,
    "--image", UI_IMAGE_URI,
    "--region", REGION,
    "--project", PROJECT_ID,
    "--platform", "managed",
    "--allow-unauthenticated",
    "--port", "8080",
    "--memory", "2Gi",
    "--max-instances", "2",
    "--set-env-vars", f"API_URL={PREDICT_URL}"
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run deploy news-credibility-ui --image europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui:v1 --region europe-west1 --project graphic-outlook-489716-n6 --platform managed --allow-unauthenticated --port 8080 --memory 2Gi --max-instances 2 --set-env-vars API_URL=https://news-credibility-api-xnipts7f7q-ew.a.run.app/predict/
Deploying container to Cloud Run service [news-credibility-ui] in project [graphic-outlook-489716-n6] region [europe-west1]
Deploying...
Setting IAM Policy..............done
Creating Revision.........................................................................................................................................................................................................................................................................................................................................................................................................

CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'run', 'deploy', 'news-credibility-ui', '--image', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui:v1', '--region', 'europe-west1', '--project', 'graphic-outlook-489716-n6', '--platform', 'managed', '--allow-unauthenticated', '--port', '8080', '--memory', '2Gi', '--max-instances', '2', '--set-env-vars', 'API_URL=https://news-credibility-api-xnipts7f7q-ew.a.run.app/predict/'], returncode=0, stdout='', stderr='Deploying container to Cloud Run service [news-credibility-ui] in project [graphic-outlook-489716-n6] region [europe-west1]\nDeploying...\nSetting IAM Policy..............done\nCreating Revision.............................................................................................................................................................................................................................................................................

## 18. Get Streamlit UI URL

This gives the public URL for the deployed Streamlit application.

In [29]:
result = run_cmd([
    GCLOUD, "run", "services", "describe", UI_SERVICE_NAME,
    "--region", REGION,
    "--project", PROJECT_ID,
    "--format", "json"
])

ui_info = json.loads(result.stdout)
UI_URL = ui_info["status"]["url"]

print("Streamlit UI URL:", UI_URL)
print("Open this URL in the browser and test a news article.")


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run services describe news-credibility-ui --region europe-west1 --project graphic-outlook-489716-n6 --format json
{
  "apiVersion": "serving.knative.dev/v1",
  "kind": "Service",
  "metadata": {
    "annotations": {
      "run.googleapis.com/client-name": "gcloud",
      "run.googleapis.com/client-version": "569.0.0",
      "run.googleapis.com/ingress": "all",
      "run.googleapis.com/ingress-status": "all",
      "run.googleapis.com/maxScale": "20",
      "run.googleapis.com/operation-id": "bbf5dabe-b74b-4cca-b7bc-b921a468a816",
      "run.googleapis.com/urls": "[\"https://news-credibility-ui-727182253496.europe-west1.run.app\",\"https://news-credibility-ui-xnipts7f7q-ew.a.run.app\"]",
      "serving.knative.dev/creator": "nishant_s1@me.iitr.ac.in",
      "serving.knative.dev/lastModifier": "nishant_s1@me.iitr.ac.in"
    },
    "creationTimestamp": "2026-05-30T16:17:54.175915Z",
    "generation": 5,
    "labels": {
      "cloud.googlea

## 19. What this deployment proves

This notebook proves the cloud deployment part of my project:

```text
local project
→ Docker image
→ Artifact Registry
→ Cloud Run API
→ Cloud Run Streamlit UI
→ public prediction endpoint
```

This is stronger than only showing a model inside a notebook because it demonstrates a real deployment workflow.

It also connects to the course topics:

- Docker and containerization,
- Artifact Registry,
- API serving,
- Cloud Run deployment,
- reproducibility,
- production-style ML serving.

### Build pipeline image

In [37]:
PIPELINE_IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/news-credibility-pipeline:v1"

In [ ]:
run_cmd([
    "docker", "build",
    "-f", "Dockerfile.gcp.pipeline",
    "-t", PIPELINE_IMAGE_URI,
    "."
])


>> docker build -f Dockerfile.gcp.pipeline -t europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline:v1 .
#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.gcp.pipeline
#1 DONE 0.0s

#1 [internal] load build definition from Dockerfile.gcp.pipeline
#1 transferring dockerfile: 529B 0.0s done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 0.5s

#3 [internal] load .dockerignore
#3 transferring context: 301B done
#3 DONE 0.0s

#4 [1/8] FROM docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0
#4 resolve docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0 0.0s done
#4 DONE 0.1s

#5 [internal] load build context
#5 transferring context: 5.07kB 0.0s done
#5 DONE 0.1s

#6 [3/8] COPY requirements.txt .
#6 CACHED

#7 [6/8] COPY src/ ./sr

CompletedProcess(args=['docker', 'build', '-f', 'Dockerfile.gcp.pipeline', '-t', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline:v1', '.'], returncode=0, stdout='', stderr='#0 building with "desktop-linux" instance using docker driver\n\n#1 [internal] load build definition from Dockerfile.gcp.pipeline\n#1 DONE 0.0s\n\n#1 [internal] load build definition from Dockerfile.gcp.pipeline\n#1 transferring dockerfile: 529B 0.0s done\n#1 DONE 0.0s\n\n#2 [internal] load metadata for docker.io/library/python:3.11-slim\n#2 DONE 0.5s\n\n#3 [internal] load .dockerignore\n#3 transferring context: 301B done\n#3 DONE 0.0s\n\n#4 [1/8] FROM docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0\n#4 resolve docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0 0.0s done\n#4 DONE 0.1s\n\n#5 [internal] load build context\n#5 transferring context: 5.0

### Push pipeline image

In [37]:
run_cmd(["docker", "build", "-f", "Dockerfile.gcp.pipeline", "-t", PIPELINE_IMAGE_URI, "."])
run_cmd(["docker", "push", PIPELINE_IMAGE_URI])


>> docker build -f Dockerfile.gcp.pipeline -t europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline:v1 .
#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.gcp.pipeline
#1 transferring dockerfile: 699B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 0.5s

#3 [internal] load .dockerignore
#3 transferring context: 303B 0.0s done
#3 DONE 0.0s

#4 [ 1/10] FROM docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0
#4 resolve docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0 0.0s done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 1.56MB 0.1s done
#5 DONE 0.1s

#6 [ 5/10] RUN pip install --no-cache-dir -r requirements.txt
#6 CACHED

#7 [ 6/10] COPY src/ ./src/
#7 CACHED

#8 [ 2/10] WORKDIR /app
#8 CACHE

CompletedProcess(args=['docker', 'push', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline:v1'], returncode=0, stdout='The push refers to repository [europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline]\n27c26afc0cfd: Waiting\n3e31c6fa7f13: Waiting\n69caaf1573d1: Waiting\n8649771fee17: Waiting\n894738e1aadb: Waiting\n0ba6a6381680: Waiting\n797d495f2c68: Waiting\ncc76a35fdefd: Waiting\n45006ceeeea9: Waiting\n5b4d6ff92fc4: Waiting\n5aea15a6b8a4: Waiting\n77ace3f26589: Waiting\n15abbcede5a9: Waiting\ned1f8bc4c55b: Waiting\n27c26afc0cfd: Waiting\n3e31c6fa7f13: Waiting\n69caaf1573d1: Waiting\n8649771fee17: Waiting\n894738e1aadb: Waiting\n0ba6a6381680: Waiting\n797d495f2c68: Waiting\ncc76a35fdefd: Waiting\n45006ceeeea9: Waiting\n5b4d6ff92fc4: Waiting\n5aea15a6b8a4: Waiting\n77ace3f26589: Waiting\n15abbcede5a9: Waiting\ned1f8bc4c55b: Waiting\n5b4d6ff92fc4: Waiting\n5aea15a6b8a4: Waiting\n77a

### Create Cloud Run Job

In [38]:
PIPELINE_JOB_NAME = "news-credibility-live-pipeline"

run_cmd([
    GCLOUD, "run", "jobs", "update", PIPELINE_JOB_NAME,
    "--image", PIPELINE_IMAGE_URI,
    "--region", REGION,
    "--project", PROJECT_ID,
    "--memory", "2Gi",
    "--tasks", "1",
    "--max-retries", "1"
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run jobs update news-credibility-live-pipeline --image europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline:v1 --region europe-west1 --project graphic-outlook-489716-n6 --memory 2Gi --tasks 1 --max-retries 1
Updating Cloud Run job [news-credibility-live-pipeline] in project [graphic-outlook-489716-n6] region [europe-west1]
Updating job...
Done.
Job [news-credibility-live-pipeline] has successfully been updated.

To execute this job, use:
gcloud run jobs execute news-credibility-live-pipeline



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'run', 'jobs', 'update', 'news-credibility-live-pipeline', '--image', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline:v1', '--region', 'europe-west1', '--project', 'graphic-outlook-489716-n6', '--memory', '2Gi', '--tasks', '1', '--max-retries', '1'], returncode=0, stdout='', stderr='Updating Cloud Run job [news-credibility-live-pipeline] in project [graphic-outlook-489716-n6] region [europe-west1]\nUpdating job...\nDone.\nJob [news-credibility-live-pipeline] has successfully been updated.\n\nTo execute this job, use:\ngcloud run jobs execute news-credibility-live-pipeline\n')

### Run the pipeline job

In [39]:
run_cmd([
    GCLOUD, "logging", "read",
    f'resource.type="cloud_run_revision"',
    "--project", PROJECT_ID,
    "--limit", "100",
    "--format", "value(textPayload)"
], check=False)


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd logging read resource.type="cloud_run_revision" --project graphic-outlook-489716-n6 --limit 100 --format value(textPayload)
  Stopping...
  URL: http://0.0.0.0:8080
  You can now view your Streamlit app in your browser.


Default STARTUP TCP probe succeeded after 1 attempt for container "news-credibility-ui-1" on port 8080.
Starting new instance. Reason: DEPLOYMENT_ROLLOUT - Instance started due to traffic shifting between revisions due to deployment, traffic split adjustment, or deployment health check.



INFO:     Finished server process [1]
INFO:     Application shutdown complete.
INFO:     Waiting for application shutdown.
INFO:     Shutting down
INFO:     169.254.169.126:16882 - "GET /predict/ HTTP/1.1" 405 Method Not Allowed


INFO:     169.254.169.126:16874 - "POST /predict HTTP/1.1" 307 Temporary Redirect

INFO:     169.254.169.126:16870 - "GET /health HTTP/1.1" 404 Not Found



Default STARTUP TCP probe succeeded after 1 attemp

CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'logging', 'read', 'resource.type="cloud_run_revision"', '--project', 'graphic-outlook-489716-n6', '--limit', '100', '--format', 'value(textPayload)'], returncode=0, stdout='  Stopping...\n  URL: http://0.0.0.0:8080\n  You can now view your Streamlit app in your browser.\n\n\nDefault STARTUP TCP probe succeeded after 1 attempt for container "news-credibility-ui-1" on port 8080.\nCollecting usage statistics. To deactivate, set browser.gatherUsageStats to false.\nStarting new instance. Reason: DEPLOYMENT_ROLLOUT - Instance started due to traffic shifting between revisions due to deployment, traffic split adjustment, or deployment health check.\n\n\n\nINFO:     Finished server process [1]\nINFO:     Application shutdown complete.\nINFO:     Waiting for application shutdown.\nINFO:     Shutting down\nINFO:     169.254.169.126:16882 - "GET /predict/ HTTP/1.1" 405 Method Not Allowed\n\n\nINFO:     169.254.169.126:168

In [40]:
run_cmd([
    GCLOUD, "run", "jobs", "execute", PIPELINE_JOB_NAME,
    "--region", REGION,
    "--project", PROJECT_ID,
    "--wait"
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run jobs execute news-credibility-live-pipeline --region europe-west1 --project graphic-outlook-489716-n6 --wait
Creating execution...
Provisioning resources...............................done
Starting execution...........................................................................................................................................................................done
Running execution.......................................................................................................................................................................................................................................................................................................................................................................done
Done.
Execution [news-credibility-live-pipeline-dp67n] has successfully completed.

View details about this execution by running:
gcloud run jobs executions describe news-credibility-

CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'run', 'jobs', 'execute', 'news-credibility-live-pipeline', '--region', 'europe-west1', '--project', 'graphic-outlook-489716-n6', '--wait'], returncode=0, stdout='', stderr='Creating execution...\nProvisioning resources...............................done\nStarting execution...........................................................................................................................................................................done\nRunning execution.......................................................................................................................................................................................................................................................................................................................................................................done\nDone.\nExecution [news-credibility-live-pipeline-dp67n] has successfully completed.\n\nView

### Check job executions

In [41]:
run_cmd([
    GCLOUD, "run", "jobs", "executions", "list",
    "--job", PIPELINE_JOB_NAME,
    "--region", REGION,
    "--project", PROJECT_ID
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run jobs executions list --job news-credibility-live-pipeline --region europe-west1 --project graphic-outlook-489716-n6
+
JOB: news-credibility-live-pipeline
EXECUTION: news-credibility-live-pipeline-dp67n
REGION: europe-west1
RUNNING: 0
COMPLETE: 1 / 1
CREATED: 2026-05-30 18:27:14 UTC
RUN BY: nishant_s1@me.iitr.ac.in

X
JOB: news-credibility-live-pipeline
EXECUTION: news-credibility-live-pipeline-4d97l
REGION: europe-west1
RUNNING: 0
COMPLETE: 0 / 1
CREATED: 2026-05-30 17:17:14 UTC
RUN BY: nishant_s1@me.iitr.ac.in

X
JOB: news-credibility-live-pipeline
EXECUTION: news-credibility-live-pipeline-z7pcr
REGION: europe-west1
RUNNING: 0
COMPLETE: 0 / 1
CREATED: 2026-05-30 16:36:58 UTC
RUN BY: nishant_s1@me.iitr.ac.in



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'run', 'jobs', 'executions', 'list', '--job', 'news-credibility-live-pipeline', '--region', 'europe-west1', '--project', 'graphic-outlook-489716-n6'], returncode=0, stdout='+\nJOB: news-credibility-live-pipeline\nEXECUTION: news-credibility-live-pipeline-dp67n\nREGION: europe-west1\nRUNNING: 0\nCOMPLETE: 1 / 1\nCREATED: 2026-05-30 18:27:14 UTC\nRUN BY: nishant_s1@me.iitr.ac.in\n\nX\nJOB: news-credibility-live-pipeline\nEXECUTION: news-credibility-live-pipeline-4d97l\nREGION: europe-west1\nRUNNING: 0\nCOMPLETE: 0 / 1\nCREATED: 2026-05-30 17:17:14 UTC\nRUN BY: nishant_s1@me.iitr.ac.in\n\nX\nJOB: news-credibility-live-pipeline\nEXECUTION: news-credibility-live-pipeline-z7pcr\nREGION: europe-west1\nRUNNING: 0\nCOMPLETE: 0 / 1\nCREATED: 2026-05-30 16:36:58 UTC\nRUN BY: nishant_s1@me.iitr.ac.in\n', stderr='')

### Get service URLs

In [42]:
run_cmd([
    GCLOUD, "run", "services", "list",
    "--region", REGION,
    "--project", PROJECT_ID
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run services list --region europe-west1 --project graphic-outlook-489716-n6
+
SERVICE: news-credibility-api
REGION: europe-west1
URL: https://news-credibility-api-727182253496.europe-west1.run.app
LAST DEPLOYED BY: nishant_s1@me.iitr.ac.in
LAST DEPLOYED AT: 2026-05-30T15:46:04.386623Z

+
SERVICE: news-credibility-ui
REGION: europe-west1
URL: https://news-credibility-ui-727182253496.europe-west1.run.app
LAST DEPLOYED BY: nishant_s1@me.iitr.ac.in
LAST DEPLOYED AT: 2026-05-30T16:22:33.755400Z



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'run', 'services', 'list', '--region', 'europe-west1', '--project', 'graphic-outlook-489716-n6'], returncode=0, stdout='+\nSERVICE: news-credibility-api\nREGION: europe-west1\nURL: https://news-credibility-api-727182253496.europe-west1.run.app\nLAST DEPLOYED BY: nishant_s1@me.iitr.ac.in\nLAST DEPLOYED AT: 2026-05-30T15:46:04.386623Z\n\n+\nSERVICE: news-credibility-ui\nREGION: europe-west1\nURL: https://news-credibility-ui-727182253496.europe-west1.run.app\nLAST DEPLOYED BY: nishant_s1@me.iitr.ac.in\nLAST DEPLOYED AT: 2026-05-30T16:22:33.755400Z\n', stderr='')

### Test API from notebook

In [44]:
run_cmd([
    GCLOUD, "run", "services", "list",
    "--region", REGION,
    "--project", PROJECT_ID
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run services list --region europe-west1 --project graphic-outlook-489716-n6
+
SERVICE: news-credibility-api
REGION: europe-west1
URL: https://news-credibility-api-727182253496.europe-west1.run.app
LAST DEPLOYED BY: nishant_s1@me.iitr.ac.in
LAST DEPLOYED AT: 2026-05-30T15:46:04.386623Z

+
SERVICE: news-credibility-ui
REGION: europe-west1
URL: https://news-credibility-ui-727182253496.europe-west1.run.app
LAST DEPLOYED BY: nishant_s1@me.iitr.ac.in
LAST DEPLOYED AT: 2026-05-30T16:22:33.755400Z



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'run', 'services', 'list', '--region', 'europe-west1', '--project', 'graphic-outlook-489716-n6'], returncode=0, stdout='+\nSERVICE: news-credibility-api\nREGION: europe-west1\nURL: https://news-credibility-api-727182253496.europe-west1.run.app\nLAST DEPLOYED BY: nishant_s1@me.iitr.ac.in\nLAST DEPLOYED AT: 2026-05-30T15:46:04.386623Z\n\n+\nSERVICE: news-credibility-ui\nREGION: europe-west1\nURL: https://news-credibility-ui-727182253496.europe-west1.run.app\nLAST DEPLOYED BY: nishant_s1@me.iitr.ac.in\nLAST DEPLOYED AT: 2026-05-30T16:22:33.755400Z\n', stderr='')

In [16]:
import requests

API_URL = "https://news-credibility-api-727182253496.europe-west1.run.app/predict/"

payload = {
    "text": "The government confirmed the new policy in an official statement."
}

response = requests.post(API_URL, json=payload)
print(response.status_code)
print(response.text)

200
{"prediction_label":"real","confidence":0.5137,"credibility_score":51,"risk_level":"Medium"}


## 20. Optional: Deploy Airflow on Google Cloud using Cloud Composer

The project already has the full pipeline deployed as a **Cloud Run Job**. This proves that the pipeline can run on Google Cloud infrastructure.

However, in the local project we also used **Airflow** for orchestration. The cloud version of Airflow on Google Cloud is called **Cloud Composer**.

In this optional section, Cloud Composer will not rerun all Python files directly. Instead, it will trigger the existing Cloud Run Job:

```text
Cloud Composer / Airflow DAG
        ↓
Cloud Run Job
        ↓
RSS ingestion → feature generation → prediction → monitoring
```

This is a cleaner cloud architecture because Airflow only orchestrates the workflow, while Cloud Run executes the containerized pipeline.

> Note: Cloud Composer can take 20–40 minutes to create and may use more credits than Cloud Run. Only run this section if you want Airflow itself visible on Google Cloud.


### 20.1 Enable Cloud Composer API

First, enable the Cloud Composer API. Cloud Composer is Google Cloud's managed Airflow service.

In [47]:
COMPOSER_ENV_NAME = "news-credibility-airflow"
COMPOSER_SA_NAME = "composer-airflow-sa"
COMPOSER_SA_EMAIL = f"{COMPOSER_SA_NAME}@{PROJECT_ID}.iam.gserviceaccount.com"

print("Composer environment:", COMPOSER_ENV_NAME)
print("Composer service account:", COMPOSER_SA_EMAIL)

Composer environment: news-credibility-airflow
Composer service account: composer-airflow-sa@graphic-outlook-489716-n6.iam.gserviceaccount.com


In [48]:
run_cmd([
    GCLOUD, "services", "enable",
    "composer.googleapis.com",
    "--project", PROJECT_ID
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd services enable composer.googleapis.com --project graphic-outlook-489716-n6


CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'services', 'enable', 'composer.googleapis.com', '--project', 'graphic-outlook-489716-n6'], returncode=0, stdout='', stderr='')

### 20.2 Create a small Cloud Composer environment

This creates a managed Airflow environment. It can take a long time, so run it only once.

If the environment already exists, this command may fail. In that case, skip to the next section.


In [49]:
run_cmd([
    GCLOUD, "iam", "service-accounts", "create", COMPOSER_SA_NAME,
    "--display-name", "Composer Airflow Service Account",
    "--project", PROJECT_ID
], check=False)


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd iam service-accounts create composer-airflow-sa --display-name Composer Airflow Service Account --project graphic-outlook-489716-n6
Created service account [composer-airflow-sa].



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'iam', 'service-accounts', 'create', 'composer-airflow-sa', '--display-name', 'Composer Airflow Service Account', '--project', 'graphic-outlook-489716-n6'], returncode=0, stdout='', stderr='Created service account [composer-airflow-sa].\n')

In [50]:
composer_roles = [
    "roles/composer.worker",
    "roles/run.developer",
    "roles/iam.serviceAccountUser",
    "roles/logging.logWriter",
    "roles/storage.objectAdmin",
]

for role in composer_roles:
    run_cmd([
        GCLOUD, "projects", "add-iam-policy-binding", PROJECT_ID,
        "--member", f"serviceAccount:{COMPOSER_SA_EMAIL}",
        "--role", role
    ], check=False)


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd projects add-iam-policy-binding graphic-outlook-489716-n6 --member serviceAccount:composer-airflow-sa@graphic-outlook-489716-n6.iam.gserviceaccount.com --role roles/composer.worker
bindings:
- members:
  - serviceAccount:service-727182253496@gcp-sa-artifactregistry.iam.gserviceaccount.com
  role: roles/artifactregistry.serviceAgent
- members:
  - serviceAccount:727182253496@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds.builder
- members:
  - serviceAccount:service-727182253496@gcp-sa-cloudbuild.iam.gserviceaccount.com
  role: roles/cloudbuild.serviceAgent
- members:
  - serviceAccount:service-727182253496@cloudcomposer-accounts.iam.gserviceaccount.com
  role: roles/composer.serviceAgent
- members:
  - serviceAccount:composer-airflow-sa@graphic-outlook-489716-n6.iam.gserviceaccount.com
  role: roles/composer.worker
- members:
  - serviceAccount:727182253496@cloudservices.gserviceaccount.com
  role: roles/compute.instanceG

In [52]:
PROJECT_NUMBER = "727182253496"

COMPOSER_SERVICE_AGENT = (
    f"service-{PROJECT_NUMBER}@cloudcomposer-accounts.iam.gserviceaccount.com"
)

run_cmd([
    GCLOUD, "projects", "add-iam-policy-binding", PROJECT_ID,
    "--member", f"serviceAccount:{COMPOSER_SERVICE_AGENT}",
    "--role", "roles/composer.ServiceAgentV2Ext"
], check=False)


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd projects add-iam-policy-binding graphic-outlook-489716-n6 --member serviceAccount:service-727182253496@cloudcomposer-accounts.iam.gserviceaccount.com --role roles/composer.ServiceAgentV2Ext
bindings:
- members:
  - serviceAccount:service-727182253496@gcp-sa-artifactregistry.iam.gserviceaccount.com
  role: roles/artifactregistry.serviceAgent
- members:
  - serviceAccount:727182253496@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds.builder
- members:
  - serviceAccount:service-727182253496@gcp-sa-cloudbuild.iam.gserviceaccount.com
  role: roles/cloudbuild.serviceAgent
- members:
  - serviceAccount:service-727182253496@cloudcomposer-accounts.iam.gserviceaccount.com
  role: roles/composer.ServiceAgentV2Ext
- members:
  - serviceAccount:service-727182253496@cloudcomposer-accounts.iam.gserviceaccount.com
  role: roles/composer.serviceAgent
- members:
  - serviceAccount:composer-airflow-sa@graphic-outlook-489716-n6.iam.gserviceac

CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'projects', 'add-iam-policy-binding', 'graphic-outlook-489716-n6', '--member', 'serviceAccount:service-727182253496@cloudcomposer-accounts.iam.gserviceaccount.com', '--role', 'roles/composer.ServiceAgentV2Ext'], returncode=0, stdout='bindings:\n- members:\n  - serviceAccount:service-727182253496@gcp-sa-artifactregistry.iam.gserviceaccount.com\n  role: roles/artifactregistry.serviceAgent\n- members:\n  - serviceAccount:727182253496@cloudbuild.gserviceaccount.com\n  role: roles/cloudbuild.builds.builder\n- members:\n  - serviceAccount:service-727182253496@gcp-sa-cloudbuild.iam.gserviceaccount.com\n  role: roles/cloudbuild.serviceAgent\n- members:\n  - serviceAccount:service-727182253496@cloudcomposer-accounts.iam.gserviceaccount.com\n  role: roles/composer.ServiceAgentV2Ext\n- members:\n  - serviceAccount:service-727182253496@cloudcomposer-accounts.iam.gserviceaccount.com\n  role: roles/composer.serviceAgent\n- m

In [53]:
run_cmd([
    GCLOUD, "composer", "environments", "create", COMPOSER_ENV_NAME,
    "--location", REGION,
    "--image-version", "composer-2-airflow-2",
    "--environment-size", "small",
    "--service-account", COMPOSER_SA_EMAIL,
    "--project", PROJECT_ID
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd composer environments create news-credibility-airflow --location europe-west1 --image-version composer-2-airflow-2 --environment-size small --service-account composer-airflow-sa@graphic-outlook-489716-n6.iam.gserviceaccount.com --project graphic-outlook-489716-n6
Waiting for [projects/graphic-outlook-489716-n6/locations/europe-west1/environments/news-credibility-airflow] to be created with [projects/graphic-outlook-489716-n6/locations/europe-west1/operations/904d9c6d-c346-4285-a624-c1665897a7c9]...
...............................................................................................................................................................................................................................................................................................................................................................................................................................................................

CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'composer', 'environments', 'create', 'news-credibility-airflow', '--location', 'europe-west1', '--image-version', 'composer-2-airflow-2', '--environment-size', 'small', '--service-account', 'composer-airflow-sa@graphic-outlook-489716-n6.iam.gserviceaccount.com', '--project', 'graphic-outlook-489716-n6'], returncode=0, stdout='', stderr='Waiting for [projects/graphic-outlook-489716-n6/locations/europe-west1/environments/news-credibility-airflow] to be created with [projects/graphic-outlook-489716-n6/locations/europe-west1/operations/904d9c6d-c346-4285-a624-c1665897a7c9]...\n.......................................................................................................................................................................................................................................................................................................................................................

### 20.3 Get the Composer DAG bucket

Cloud Composer stores DAG files in a Google Cloud Storage bucket. We need the bucket path so that we can upload our Airflow DAG.

In [59]:
result = run_cmd([
    GCLOUD, "composer", "environments", "describe", COMPOSER_ENV_NAME,
    "--location", REGION,
    "--project", PROJECT_ID,
    "--format", "value(config.dagGcsPrefix)"
])

DAG_GCS_PREFIX = result.stdout.strip()
print("Composer DAG folder:", DAG_GCS_PREFIX)


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd composer environments describe news-credibility-airflow --location europe-west1 --project graphic-outlook-489716-n6 --format value(config.dagGcsPrefix)
gs://europe-west1-news-credibili-904d9c6d-bucket/dags

Composer DAG folder: gs://europe-west1-news-credibili-904d9c6d-bucket/dags


### 20.4 Create a Cloud Composer DAG that triggers the Cloud Run Job

This DAG runs inside Cloud Composer. It triggers the already deployed Cloud Run Job named `news-credibility-live-pipeline`.

This gives a cloud Airflow view of the pipeline while keeping the actual pipeline execution inside Cloud Run.


In [73]:
from pathlib import Path

dag_code = f'''
from datetime import datetime

from airflow import DAG
from airflow.providers.google.cloud.operators.cloud_run import CloudRunExecuteJobOperator

PROJECT_ID = "{PROJECT_ID}"
REGION = "{REGION}"
JOB_NAME = "{PIPELINE_JOB_NAME}"

with DAG(
    dag_id="news_credibility_cloud_pipeline",
    start_date=datetime(2026, 5, 30),
    schedule_interval="@daily",
    catchup=False,
    tags=["mlops", "news-credibility", "cloud-run"],
) as dag:

    run_cloud_pipeline = CloudRunExecuteJobOperator(
        task_id="run_cloud_run_pipeline_job",
        project_id=PROJECT_ID,
        region=REGION,
        job_name=JOB_NAME,
    )
'''

dag_path = Path("news_credibility_cloud_pipeline.py")
dag_path.write_text(dag_code)

print("Created DAG:", dag_path.resolve())

Created DAG: D:\_HSLU\MLOPS\real-time-news-credibility\news_credibility_cloud_pipeline.py


### 20.5 Upload the DAG to Cloud Composer

After uploading, open the Airflow web UI from the Google Cloud Composer page. The DAG should appear as:

```text
news_credibility_cloud_pipeline
```

When triggered, it will execute the Cloud Run Job that runs the full live pipeline.


In [61]:
run_cmd([
    GCLOUD, "storage", "cp",
    "news_credibility_cloud_pipeline.py",
    DAG_GCS_PREFIX + "/news_credibility_cloud_pipeline.py"
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd storage cp news_credibility_cloud_pipeline.py gs://europe-west1-news-credibili-904d9c6d-bucket/dags/news_credibility_cloud_pipeline.py
Copying file://news_credibility_cloud_pipeline.py to gs://europe-west1-news-credibili-904d9c6d-bucket/dags/news_credibility_cloud_pipeline.py
  
....



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'storage', 'cp', 'news_credibility_cloud_pipeline.py', 'gs://europe-west1-news-credibili-904d9c6d-bucket/dags/news_credibility_cloud_pipeline.py'], returncode=0, stdout='', stderr='Copying file://news_credibility_cloud_pipeline.py to gs://europe-west1-news-credibili-904d9c6d-bucket/dags/news_credibility_cloud_pipeline.py\n  \n....\n')

### 20.6 Verify Composer environment and DAG upload

Use these commands to check the Composer environment and confirm where the DAG was uploaded.

In [62]:
run_cmd([
    GCLOUD, "composer", "environments", "list",
    "--locations", REGION,
    "--project", PROJECT_ID
])

print("DAG uploaded to:", DAG_GCS_PREFIX + "/news_credibility_cloud_pipeline.py")
print("Open Google Cloud Console → Composer → news-credibility-airflow → Airflow UI")


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd composer environments list --locations europe-west1 --project graphic-outlook-489716-n6
NAME: news-credibility-airflow
LOCATION: europe-west1
STATE: RUNNING
CREATE_TIME: 2026-05-31T21:13:55.515415Z

DAG uploaded to: gs://europe-west1-news-credibili-904d9c6d-bucket/dags/news_credibility_cloud_pipeline.py
Open Google Cloud Console → Composer → news-credibility-airflow → Airflow UI


### 20.7 What this proves

With this setup, the cloud architecture becomes:

```text
Cloud Composer / Airflow
        ↓
Cloud Run Job
        ↓
RSS ingestion
        ↓
Feature engineering
        ↓
Live prediction
        ↓
Monitoring reports
```

This is stronger than only deploying the trained model because the full live pipeline is orchestrated on Google Cloud.

For the presentation/report, I can say:

> The full live pipeline was deployed on Google Cloud. FastAPI and Streamlit run as Cloud Run services, the container images are stored in Artifact Registry, and the live ingestion → feature generation → prediction → monitoring workflow runs as a Cloud Run Job. Optionally, Cloud Composer is used as managed Airflow to trigger the Cloud Run Job on a schedule.

### Get Airflow UI URL

In [64]:
run_cmd([
    GCLOUD, "composer", "environments", "describe", COMPOSER_ENV_NAME,
    "--location", REGION,
    "--project", PROJECT_ID,
    "--format", "value(config.airflowUri)"
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd composer environments describe news-credibility-airflow --location europe-west1 --project graphic-outlook-489716-n6 --format value(config.airflowUri)
https://dc3ebd18045944bbaa241a70fe9c6936-dot-europe-west1.composer.googleusercontent.com



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'composer', 'environments', 'describe', 'news-credibility-airflow', '--location', 'europe-west1', '--project', 'graphic-outlook-489716-n6', '--format', 'value(config.airflowUri)'], returncode=0, stdout='https://dc3ebd18045944bbaa241a70fe9c6936-dot-europe-west1.composer.googleusercontent.com\n', stderr='')

# Deploy MLflow UI to Google Cloud Run

In the main project, MLflow is used for local experiment tracking and model registry evidence.  
For the cloud demo, we can also deploy a lightweight MLflow UI as a separate Cloud Run service.

This is useful to show that MLflow is part of the MLOps stack.  
However, this simple version uses local container storage, so it is mainly for demonstration.  
For production, MLflow should use Cloud SQL as the backend store and Google Cloud Storage as the artifact store.


In [67]:
MLFLOW_SERVICE_NAME = "news-credibility-mlflow"
MLFLOW_IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/news-credibility-mlflow:v1"

print("MLflow service:", MLFLOW_SERVICE_NAME)
print("MLflow image:", MLFLOW_IMAGE_URI)

MLflow service: news-credibility-mlflow
MLflow image: europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-mlflow:v1


## Create a Dockerfile for MLflow

This Dockerfile starts the MLflow UI on port `8080`, which is the default port expected by Cloud Run.


In [68]:
from pathlib import Path

mlflow_dockerfile = r"""
FROM python:3.11-slim

WORKDIR /app

RUN pip install --upgrade pip
RUN pip install mlflow==2.14.1

RUN mkdir -p /app/mlruns

EXPOSE 8080

CMD ["mlflow", "ui", "--backend-store-uri", "/app/mlruns", "--host", "0.0.0.0", "--port", "8080"]
"""

Path("Dockerfile.gcp.mlflow").write_text(mlflow_dockerfile.strip() + "\n")
print("Created Dockerfile.gcp.mlflow")

Created Dockerfile.gcp.mlflow


## Build the MLflow Docker image

In [69]:
run_cmd([
    "docker", "build",
    "-f", "Dockerfile.gcp.mlflow",
    "-t", MLFLOW_IMAGE_URI,
    "."
])


>> docker build -f Dockerfile.gcp.mlflow -t europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-mlflow:v1 .


Exception in thread Thread-119 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\LENOVO\anaconda3\Lib\threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "C:\Users\LENOVO\anaconda3\Lib\threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\LENOVO\anaconda3\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "C:\Users\LENOVO\anaconda3\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 1095: character maps to <undefined>


CompletedProcess(args=['docker', 'build', '-f', 'Dockerfile.gcp.mlflow', '-t', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-mlflow:v1', '.'], returncode=0, stdout='')

## Push the MLflow image to Artifact Registry

In [70]:
run_cmd([
    "docker", "push",
    MLFLOW_IMAGE_URI
])



>> docker push europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-mlflow:v1
The push refers to repository [europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-mlflow]
b32c1e70e390: Waiting
20074fd3c284: Waiting
797d495f2c68: Waiting
3e31c6fa7f13: Waiting
1d9ea3733389: Waiting
5b4d6ff92fc4: Waiting
45006ceeeea9: Waiting
8649771fee17: Waiting
afb9b541f32c: Waiting
b32c1e70e390: Waiting
20074fd3c284: Waiting
797d495f2c68: Waiting
3e31c6fa7f13: Waiting
1d9ea3733389: Waiting
5b4d6ff92fc4: Waiting
45006ceeeea9: Waiting
8649771fee17: Waiting
afb9b541f32c: Waiting
5b4d6ff92fc4: Waiting
45006ceeeea9: Waiting
8649771fee17: Waiting
afb9b541f32c: Waiting
b32c1e70e390: Waiting
20074fd3c284: Waiting
797d495f2c68: Waiting
3e31c6fa7f13: Waiting
1d9ea3733389: Waiting
5b4d6ff92fc4: Waiting
45006ceeeea9: Waiting
8649771fee17: Waiting
afb9b541f32c: Waiting
b32c1e70e390: Waiting
20074fd3c284: Waiting
797d495f2c68: Waiting
3

CompletedProcess(args=['docker', 'push', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-mlflow:v1'], returncode=0, stdout='The push refers to repository [europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-mlflow]\nb32c1e70e390: Waiting\n20074fd3c284: Waiting\n797d495f2c68: Waiting\n3e31c6fa7f13: Waiting\n1d9ea3733389: Waiting\n5b4d6ff92fc4: Waiting\n45006ceeeea9: Waiting\n8649771fee17: Waiting\nafb9b541f32c: Waiting\nb32c1e70e390: Waiting\n20074fd3c284: Waiting\n797d495f2c68: Waiting\n3e31c6fa7f13: Waiting\n1d9ea3733389: Waiting\n5b4d6ff92fc4: Waiting\n45006ceeeea9: Waiting\n8649771fee17: Waiting\nafb9b541f32c: Waiting\n5b4d6ff92fc4: Waiting\n45006ceeeea9: Waiting\n8649771fee17: Waiting\nafb9b541f32c: Waiting\nb32c1e70e390: Waiting\n20074fd3c284: Waiting\n797d495f2c68: Waiting\n3e31c6fa7f13: Waiting\n1d9ea3733389: Waiting\n5b4d6ff92fc4: Waiting\n45006ceeeea9: Waiting\n8649771fee17: Waiting\nafb9b54

## Deploy MLflow to Cloud Run


In [71]:
run_cmd([
    GCLOUD, "run", "deploy", MLFLOW_SERVICE_NAME,
    "--image", MLFLOW_IMAGE_URI,
    "--region", REGION,
    "--project", PROJECT_ID,
    "--platform", "managed",
    "--allow-unauthenticated",
    "--port", "8080",
    "--memory", "1Gi",
    "--max-instances", "1"
])



>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run deploy news-credibility-mlflow --image europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-mlflow:v1 --region europe-west1 --project graphic-outlook-489716-n6 --platform managed --allow-unauthenticated --port 8080 --memory 1Gi --max-instances 1
Deploying container to Cloud Run service [news-credibility-mlflow] in project [graphic-outlook-489716-n6] region [europe-west1]
Deploying new service...
Setting IAM Policy..............done
Creating Revision..............................................................................................................................................................................................................................................................................................................................................................................................................................done
Routing traffic.....done
Done.
Serv

CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'run', 'deploy', 'news-credibility-mlflow', '--image', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-mlflow:v1', '--region', 'europe-west1', '--project', 'graphic-outlook-489716-n6', '--platform', 'managed', '--allow-unauthenticated', '--port', '8080', '--memory', '1Gi', '--max-instances', '1'], returncode=0, stdout='', stderr='Deploying container to Cloud Run service [news-credibility-mlflow] in project [graphic-outlook-489716-n6] region [europe-west1]\nDeploying new service...\nSetting IAM Policy..............done\nCreating Revision................................................................................................................................................................................................................................................................................................................................................

## Get the MLflow Cloud Run URL


In [72]:
result = run_cmd([
    GCLOUD, "run", "services", "describe", MLFLOW_SERVICE_NAME,
    "--region", REGION,
    "--project", PROJECT_ID,
    "--format", "value(status.url)"
])

MLFLOW_URL = result.stdout.strip()
print("MLflow URL:", MLFLOW_URL)



>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run services describe news-credibility-mlflow --region europe-west1 --project graphic-outlook-489716-n6 --format value(status.url)
https://news-credibility-mlflow-xnipts7f7q-ew.a.run.app

MLflow URL: https://news-credibility-mlflow-xnipts7f7q-ew.a.run.app


## Important note

This Cloud Run MLflow deployment is a lightweight demo.  
Because Cloud Run storage is ephemeral, runs saved only inside the container may disappear when the instance restarts.

For a production setup, the better architecture is:

```text
MLflow Tracking Server on Cloud Run
+ Cloud SQL for metadata
+ Google Cloud Storage for artifacts
```

For this project, the local MLflow runs remain the main evidence for experiment tracking, while this Cloud Run deployment demonstrates that MLflow can also be containerized and exposed as part of the cloud MLOps stack.


# Connecting cloud streamlit with cloud run jon using GCS

In [114]:
GCS_BUCKET = "news-credibility-live-results-graphic-outlook"

run_cmd([
    GCLOUD, "storage", "buckets", "create",
    f"gs://{GCS_BUCKET}",
    "--location", REGION,
    "--project", PROJECT_ID
], check=False)


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd storage buckets create gs://news-credibility-live-results-graphic-outlook --location europe-west1 --project graphic-outlook-489716-n6
Creating gs://news-credibility-live-results-graphic-outlook/...
ERROR: (gcloud.storage.buckets.create) HTTPError 409: Your previous request to create the named bucket succeeded and you already own it.



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'storage', 'buckets', 'create', 'gs://news-credibility-live-results-graphic-outlook', '--location', 'europe-west1', '--project', 'graphic-outlook-489716-n6'], returncode=1, stdout='', stderr='Creating gs://news-credibility-live-results-graphic-outlook/...\nERROR: (gcloud.storage.buckets.create) HTTPError 409: Your previous request to create the named bucket succeeded and you already own it.\n')

## Rebuild pipeline image

In [115]:
PIPELINE_IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/news-credibility-pipeline:v2"

run_cmd([
    "docker", "build",
    "-f", "Dockerfile.gcp.pipeline",
    "-t", PIPELINE_IMAGE_URI,
    "."
])


>> docker build -f Dockerfile.gcp.pipeline -t europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline:v2 .
#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.gcp.pipeline
#1 transferring dockerfile: 699B 0.0s done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 0.6s

#3 [internal] load .dockerignore
#3 transferring context: 303B done
#3 DONE 0.0s

#4 [internal] load build context
#4 DONE 0.0s

#5 [ 1/10] FROM docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0
#5 resolve docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0 0.1s done
#5 DONE 0.1s

#4 [internal] load build context
#4 transferring context: 1.57MB 0.1s done
#4 DONE 0.1s

#6 [ 3/10] COPY requirements.txt .
#6 CACHED

#7 [ 4/10] RUN pip install --upgrade pip
#7 CACHED


CompletedProcess(args=['docker', 'build', '-f', 'Dockerfile.gcp.pipeline', '-t', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline:v2', '.'], returncode=0, stdout='', stderr='#0 building with "desktop-linux" instance using docker driver\n\n#1 [internal] load build definition from Dockerfile.gcp.pipeline\n#1 transferring dockerfile: 699B 0.0s done\n#1 DONE 0.0s\n\n#2 [internal] load metadata for docker.io/library/python:3.11-slim\n#2 DONE 0.6s\n\n#3 [internal] load .dockerignore\n#3 transferring context: 303B done\n#3 DONE 0.0s\n\n#4 [internal] load build context\n#4 DONE 0.0s\n\n#5 [ 1/10] FROM docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0\n#5 resolve docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0 0.1s done\n#5 DONE 0.1s\n\n#4 [internal] load build context\n#4 transferring context: 1.57MB 0.1s done\n#4 DONE 0.1s\n\

## Push pipeline image

In [116]:
run_cmd([
    "docker", "push",
    PIPELINE_IMAGE_URI
])


>> docker push europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline:v2
The push refers to repository [europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline]
5b4d6ff92fc4: Waiting
76231b579fd9: Waiting
48db0bad2b49: Waiting
1b6786250204: Waiting
45006ceeeea9: Waiting
1cf68de0fe92: Waiting
3f1b0d4ed7bc: Waiting
69caaf1573d1: Waiting
5aea15a6b8a4: Waiting
0ba6a6381680: Waiting
8649771fee17: Waiting
3e31c6fa7f13: Waiting
797d495f2c68: Waiting
8eb5af9829b5: Waiting
5b4d6ff92fc4: Waiting
76231b579fd9: Waiting
48db0bad2b49: Waiting
1b6786250204: Waiting
45006ceeeea9: Waiting
1cf68de0fe92: Waiting
3f1b0d4ed7bc: Waiting
69caaf1573d1: Waiting
5aea15a6b8a4: Waiting
0ba6a6381680: Waiting
8649771fee17: Waiting
3e31c6fa7f13: Waiting
797d495f2c68: Waiting
8eb5af9829b5: Waiting
1cf68de0fe92: Waiting
3f1b0d4ed7bc: Waiting
69caaf1573d1: Waiting
5aea15a6b8a4: Waiting
0ba6a6381680: Waiting
8649771fee17: Waiti

CompletedProcess(args=['docker', 'push', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline:v2'], returncode=0, stdout='The push refers to repository [europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline]\n5b4d6ff92fc4: Waiting\n76231b579fd9: Waiting\n48db0bad2b49: Waiting\n1b6786250204: Waiting\n45006ceeeea9: Waiting\n1cf68de0fe92: Waiting\n3f1b0d4ed7bc: Waiting\n69caaf1573d1: Waiting\n5aea15a6b8a4: Waiting\n0ba6a6381680: Waiting\n8649771fee17: Waiting\n3e31c6fa7f13: Waiting\n797d495f2c68: Waiting\n8eb5af9829b5: Waiting\n5b4d6ff92fc4: Waiting\n76231b579fd9: Waiting\n48db0bad2b49: Waiting\n1b6786250204: Waiting\n45006ceeeea9: Waiting\n1cf68de0fe92: Waiting\n3f1b0d4ed7bc: Waiting\n69caaf1573d1: Waiting\n5aea15a6b8a4: Waiting\n0ba6a6381680: Waiting\n8649771fee17: Waiting\n3e31c6fa7f13: Waiting\n797d495f2c68: Waiting\n8eb5af9829b5: Waiting\n1cf68de0fe92: Waiting\n3f1b0d4ed7bc: Waiting\n69c

## Update Cloud Run Job with GCS env var

In [117]:
run_cmd([
    GCLOUD, "run", "jobs", "update", PIPELINE_JOB_NAME,
    "--image", PIPELINE_IMAGE_URI,
    "--region", REGION,
    "--project", PROJECT_ID,
  
    "--set-env-vars", f"GCS_BUCKET={GCS_BUCKET}"
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run jobs update news-credibility-live-pipeline --image europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline:v2 --region europe-west1 --project graphic-outlook-489716-n6 --set-env-vars GCS_BUCKET=news-credibility-live-results-graphic-outlook
Updating Cloud Run job [news-credibility-live-pipeline] in project [graphic-outlook-489716-n6] region [europe-west1]
Updating job...
Done.
Job [news-credibility-live-pipeline] has successfully been updated.

To execute this job, use:
gcloud run jobs execute news-credibility-live-pipeline



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'run', 'jobs', 'update', 'news-credibility-live-pipeline', '--image', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline:v2', '--region', 'europe-west1', '--project', 'graphic-outlook-489716-n6', '--set-env-vars', 'GCS_BUCKET=news-credibility-live-results-graphic-outlook'], returncode=0, stdout='', stderr='Updating Cloud Run job [news-credibility-live-pipeline] in project [graphic-outlook-489716-n6] region [europe-west1]\nUpdating job...\nDone.\nJob [news-credibility-live-pipeline] has successfully been updated.\n\nTo execute this job, use:\ngcloud run jobs execute news-credibility-live-pipeline\n')

## Run pipeline job

In [118]:
run_cmd([
    GCLOUD, "run", "jobs", "execute", PIPELINE_JOB_NAME,
    "--region", REGION,
    "--project", PROJECT_ID,
    "--wait"
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run jobs execute news-credibility-live-pipeline --region europe-west1 --project graphic-outlook-489716-n6 --wait
Creating execution...
Provisioning resources..........................................done
Starting execution.........................................................................................................................................................................................................................................................................................................................................done
Running execution..........................................................................................................................................................................................................................................................................................................................................................................................

CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'run', 'jobs', 'execute', 'news-credibility-live-pipeline', '--region', 'europe-west1', '--project', 'graphic-outlook-489716-n6', '--wait'], returncode=0, stdout='', stderr='Creating execution...\nProvisioning resources..........................................done\nStarting execution.........................................................................................................................................................................................................................................................................................................................................done\nRunning execution...............................................................................................................................................................................................................................................................................................

In [119]:
run_cmd([
    GCLOUD, "run", "jobs", "executions", "describe",
    "news-credibility-live-pipeline-zz8p4",
    "--region", REGION,
    "--project", PROJECT_ID
], check=False)


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run jobs executions describe news-credibility-live-pipeline-zz8p4 --region europe-west1 --project graphic-outlook-489716-n6
X Execution news-credibility-live-pipeline-zz8p4 in region europe-west1
0 tasks completed successfully
1 task failed to complete
Elapsed time: 1 minute and 56 seconds
 
Log URI: https://console.cloud.google.com/logs/viewer?project=graphic-outlook-489716-n6&advancedFilter=resource.type%3D%22cloud_run_job%22%0Aresource.labels.job_name%3D%22news-credibility-live-pipeline%22%0Aresource.labels.location%3D%22europe-west1%22%0Alabels.%22run.googleapis.com/execution_name%22%3D%22news-credibility-live-pipeline-zz8p4%22
 
Tasks:           1
Parallelism:     1
Container None
  Image:         europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline@sha256:7328e37be9550b9da00e28178f55ee70e27d37e919dd505930052f9f7633feba
  Memory:        2Gi
  CPU:           1000m
  Env vars:
    GCS_B

CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'run', 'jobs', 'executions', 'describe', 'news-credibility-live-pipeline-zz8p4', '--region', 'europe-west1', '--project', 'graphic-outlook-489716-n6'], returncode=0, stdout='X Execution news-credibility-live-pipeline-zz8p4 in region europe-west1\n0 tasks completed successfully\n1 task failed to complete\nElapsed time: 1 minute and 56 seconds\n \nLog URI: https://console.cloud.google.com/logs/viewer?project=graphic-outlook-489716-n6&advancedFilter=resource.type%3D%22cloud_run_job%22%0Aresource.labels.job_name%3D%22news-credibility-live-pipeline%22%0Aresource.labels.location%3D%22europe-west1%22%0Alabels.%22run.googleapis.com/execution_name%22%3D%22news-credibility-live-pipeline-zz8p4%22\n \nTasks:           1\nParallelism:     1\nContainer None\n  Image:         europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-pipeline@sha256:7328e37be9550b9da00e28178f55ee70e27d37e919d

## Check files in GCS

In [120]:
run_cmd([
    GCLOUD, "storage", "ls",
    f"gs://{GCS_BUCKET}/**"
], check=False)


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd storage ls gs://news-credibility-live-results-graphic-outlook/**
gs://news-credibility-live-results-graphic-outlook/live/live_news_predictions.csv
gs://news-credibility-live-results-graphic-outlook/live/live_news_predictions.parquet



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'storage', 'ls', 'gs://news-credibility-live-results-graphic-outlook/**'], returncode=0, stdout='gs://news-credibility-live-results-graphic-outlook/live/live_news_predictions.csv\ngs://news-credibility-live-results-graphic-outlook/live/live_news_predictions.parquet\n', stderr='')

## Rebuild UI image

In [121]:
UI_IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/news-credibility-ui:v3"

run_cmd([
    "docker", "build",
    "-f", "Dockerfile.gcp.ui",
    "-t", UI_IMAGE_URI,
    "."
])


>> docker build -f Dockerfile.gcp.ui -t europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui:v3 .
#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.gcp.ui
#1 transferring dockerfile: 521B 0.0s done
#1 DONE 0.0s

#2 [auth] library/python:pull token for registry-1.docker.io
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 DONE 0.8s

#4 [internal] load .dockerignore
#4 transferring context: 303B done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 4.07kB 0.0s done
#5 DONE 0.1s

#6 [1/8] FROM docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0
#6 resolve docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0
#6 resolve docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e4

CompletedProcess(args=['docker', 'build', '-f', 'Dockerfile.gcp.ui', '-t', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui:v3', '.'], returncode=0, stdout='', stderr='#0 building with "desktop-linux" instance using docker driver\n\n#1 [internal] load build definition from Dockerfile.gcp.ui\n#1 transferring dockerfile: 521B 0.0s done\n#1 DONE 0.0s\n\n#2 [auth] library/python:pull token for registry-1.docker.io\n#2 DONE 0.0s\n\n#3 [internal] load metadata for docker.io/library/python:3.11-slim\n#3 DONE 0.8s\n\n#4 [internal] load .dockerignore\n#4 transferring context: 303B done\n#4 DONE 0.0s\n\n#5 [internal] load build context\n#5 transferring context: 4.07kB 0.0s done\n#5 DONE 0.1s\n\n#6 [1/8] FROM docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0\n#6 resolve docker.io/library/python:3.11-slim@sha256:a3ab0b966bc4e91546a033e22093cb840908979487a9fc0e6e38295747e49ac0\n#6 resolve docker

## Push UI image

In [122]:
run_cmd([
    "docker", "push",
    UI_IMAGE_URI
])


>> docker push europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui:v3
The push refers to repository [europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui]
3e31c6fa7f13: Waiting
69caaf1573d1: Waiting
5aea15a6b8a4: Waiting
45006ceeeea9: Waiting
9c4a7a37c9dc: Waiting
0ba6a6381680: Waiting
8649771fee17: Waiting
8eb5af9829b5: Waiting
48db0bad2b49: Waiting
797d495f2c68: Waiting
5b4d6ff92fc4: Waiting
76231b579fd9: Waiting
48db0bad2b49: Waiting
797d495f2c68: Waiting
5b4d6ff92fc4: Waiting
76231b579fd9: Waiting
3e31c6fa7f13: Waiting
69caaf1573d1: Waiting
5aea15a6b8a4: Waiting
45006ceeeea9: Waiting
9c4a7a37c9dc: Waiting
0ba6a6381680: Waiting
8649771fee17: Waiting
8eb5af9829b5: Waiting
9c4a7a37c9dc: Waiting
0ba6a6381680: Waiting
8649771fee17: Waiting
8eb5af9829b5: Waiting
48db0bad2b49: Waiting
797d495f2c68: Waiting
5b4d6ff92fc4: Waiting
76231b579fd9: Waiting
3e31c6fa7f13: Waiting
69caaf1573d1: Waiting
5aea15a6b

CompletedProcess(args=['docker', 'push', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui:v3'], returncode=0, stdout='The push refers to repository [europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui]\n3e31c6fa7f13: Waiting\n69caaf1573d1: Waiting\n5aea15a6b8a4: Waiting\n45006ceeeea9: Waiting\n9c4a7a37c9dc: Waiting\n0ba6a6381680: Waiting\n8649771fee17: Waiting\n8eb5af9829b5: Waiting\n48db0bad2b49: Waiting\n797d495f2c68: Waiting\n5b4d6ff92fc4: Waiting\n76231b579fd9: Waiting\n48db0bad2b49: Waiting\n797d495f2c68: Waiting\n5b4d6ff92fc4: Waiting\n76231b579fd9: Waiting\n3e31c6fa7f13: Waiting\n69caaf1573d1: Waiting\n5aea15a6b8a4: Waiting\n45006ceeeea9: Waiting\n9c4a7a37c9dc: Waiting\n0ba6a6381680: Waiting\n8649771fee17: Waiting\n8eb5af9829b5: Waiting\n9c4a7a37c9dc: Waiting\n0ba6a6381680: Waiting\n8649771fee17: Waiting\n8eb5af9829b5: Waiting\n48db0bad2b49: Waiting\n797d495f2c68: Waiting\n5b4d6ff92fc4: W

## Redeploy UI with API + GCS env vars

In [123]:
run_cmd([
    GCLOUD, "run", "deploy", UI_SERVICE_NAME,
    "--image", UI_IMAGE_URI,
    "--region", REGION,
    "--project", PROJECT_ID,
    "--allow-unauthenticated",
    "--port", "8080",
    "--memory", "2Gi",
    "--set-env-vars",
    f"API_URL={PREDICT_URL},GCS_BUCKET={GCS_BUCKET}"
])


>> D:\_HSLU\GCL\google-cloud-sdk\bin\gcloud.cmd run deploy news-credibility-ui --image europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui:v3 --region europe-west1 --project graphic-outlook-489716-n6 --allow-unauthenticated --port 8080 --memory 2Gi --set-env-vars API_URL=https://news-credibility-api-xnipts7f7q-ew.a.run.app/predict/,GCS_BUCKET=news-credibility-live-results-graphic-outlook
Deploying container to Cloud Run service [news-credibility-ui] in project [graphic-outlook-489716-n6] region [europe-west1]
Deploying...
Setting IAM Policy..............done
Creating Revision...........................................................................done
Routing traffic.....done
Done.
Service [news-credibility-ui] revision [news-credibility-ui-00009-brf] has been deployed and is serving 100 percent of traffic.
Service URL: https://news-credibility-ui-727182253496.europe-west1.run.app



CompletedProcess(args=['D:\\_HSLU\\GCL\\google-cloud-sdk\\bin\\gcloud.cmd', 'run', 'deploy', 'news-credibility-ui', '--image', 'europe-west1-docker.pkg.dev/graphic-outlook-489716-n6/news-credibility-repo/news-credibility-ui:v3', '--region', 'europe-west1', '--project', 'graphic-outlook-489716-n6', '--allow-unauthenticated', '--port', '8080', '--memory', '2Gi', '--set-env-vars', 'API_URL=https://news-credibility-api-xnipts7f7q-ew.a.run.app/predict/,GCS_BUCKET=news-credibility-live-results-graphic-outlook'], returncode=0, stdout='', stderr='Deploying container to Cloud Run service [news-credibility-ui] in project [graphic-outlook-489716-n6] region [europe-west1]\nDeploying...\nSetting IAM Policy..............done\nCreating Revision...........................................................................done\nRouting traffic.....done\nDone.\nService [news-credibility-ui] revision [news-credibility-ui-00009-brf] has been deployed and is serving 100 percent of traffic.\nService URL: https

## 20. Useful debugging commands

If something fails, these commands help inspect the deployed services.

In [65]:
print("API logs:")
print(f"gcloud run services logs read {API_SERVICE_NAME} --region {REGION} --project {PROJECT_ID} --limit 50")

print("\nUI logs:")
print(f"gcloud run services logs read {UI_SERVICE_NAME} --region {REGION} --project {PROJECT_ID} --limit 50")

print("\nList Cloud Run services:")
print(f"gcloud run services list --region {REGION} --project {PROJECT_ID}")

API logs:
gcloud run services logs read news-credibility-api --region europe-west1 --project graphic-outlook-489716-n6 --limit 50

UI logs:
gcloud run services logs read news-credibility-ui --region europe-west1 --project graphic-outlook-489716-n6 --limit 50

List Cloud Run services:
gcloud run services list --region europe-west1 --project graphic-outlook-489716-n6


In [ ]:
# Delete Cloud Run services
# run_cmd(["gcloud", "run", "services", "delete", API_SERVICE_NAME, "--region", REGION, "--project", PROJECT_ID, "--quiet"])
# run_cmd(["gcloud", "run", "services", "delete", UI_SERVICE_NAME, "--region", REGION, "--project", PROJECT_ID, "--quiet"])

# Delete Docker image from Artifact Registry
# run_cmd(["gcloud", "artifacts", "docker", "images", "delete", IMAGE_URI, "--project", PROJECT_ID, "--quiet"])